## ETE Simple RNN Implementation

In [39]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
# IMDB dataset from tensorflow
from tensorflow.keras.datasets import imdb


In [40]:
# Loading the dataset
max_features = 10000 # vocabulary size
(X_train,y_train),(X_test,y_test) = imdb.load_data(num_words=max_features)

c:\Users\sharm\anaconda3\envs\ai-ml\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


In [41]:
# reviewing sample
sample_review = X_train[0]
sample_label =  y_train[0]

In [42]:
# mapping words index back to words
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key ,value in word_index.items()}

In [43]:
# Padding 
from tensorflow.keras.preprocessing import sequence

max_len = 500 # vocab size
X_train = sequence.pad_sequences(X_train,maxlen=max_len)
X_test = sequence.pad_sequences(X_test,maxlen=max_len)



In [44]:
# training simple RNN
model=Sequential()
model.add(Embedding(max_features,128, input_length=max_len))
model.add(SimpleRNN(128,activation='relu'))
model.add(Dense(1,activation='sigmoid'))
model.build(input_shape=(None, max_len))

c:\Users\sharm\anaconda3\envs\ai-ml\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [45]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [46]:
# compiling for assigning optimizer and loss
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [47]:
# Creating instance ; early stopping
from tensorflow.keras.callbacks import EarlyStopping
earlystopping=EarlyStopping(monitor = 'val_loss',patience =5,restore_best_weights =True)

In [48]:
# Training with early stopping
history=model.fit(
    X_train,y_train,epochs=10,batch_size =32,
    validation_split =0.2,
    callbacks = [earlystopping]
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 27s 41ms/step - accuracy: 0.5893 - loss: 990105174016.0000 - val_accuracy: 0.6062 - val_loss: 0.6575
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.6707 - loss: 0.6204 - val_accuracy: 0.6128 - val_loss: 0.6865
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 32s 51ms/step - accuracy: 0.7524 - loss: 1.1375 - val_accuracy: 0.6624 - val_loss: 0.5970
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 34s 54ms/step - accuracy: 0.7996 - loss: 0.4524 - val_accuracy: 0.7798 - val_loss: 0.4763
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 36s 57ms/step - accuracy: 0.7330 - loss: 55851.0078 - val_accuracy: 0.6994 - val_loss: 0.5995
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 37s 59ms/step - accuracy: 0.8077 - loss: 0.4202 - val_accuracy: 0.7302 - val_loss: 0.5576
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 34s 55ms/step - accuracy: 0.8361 - loss: 0.3736 - val_accuracy: 0.7326 - val_loss: 0.5541
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.8643 - 

In [49]:
# saving the model
model.save('simplernn_imdb.h5')

### Prediction

In [50]:
# weights
model.get_weights()

[array([[-3.5769227e-03, -3.3301204e-02,  4.2749502e-02, ...,
         -4.6962124e-01,  1.2597156e-01,  3.4177005e-02],
        [-5.6343712e-03, -4.4173133e-02, -3.8288769e-04, ...,
          1.9662175e-02, -6.4861640e-02, -9.9070081e-03],
        [-5.0071105e-02, -1.7305353e-01,  7.7765062e-02, ...,
         -3.0118420e-03, -5.5278111e-02, -7.5822920e-02],
        ...,
        [-2.0189218e-03,  6.9253698e-02, -1.3724474e-02, ...,
          8.5415497e-02, -6.1906442e-02,  1.8784089e-02],
        [ 7.9208910e-02,  1.6554486e-02, -2.8329963e-02, ...,
         -6.0876507e-02,  5.6584001e-02,  3.8169671e-02],
        [ 5.3039923e-02,  3.2448292e-02, -4.2819470e-02, ...,
          4.7176946e-02, -5.4532439e-02,  2.3241702e-02]],
       shape=(10000, 128), dtype=float32),
 array([[ 0.09275164, -0.14689803, -0.02081958, ...,  0.15406911,
         -0.08422501, -0.06085466],
        [-0.05145431, -0.07801782, -0.03935546, ...,  0.00328868,
          0.04769043, -0.00851736],
        [-0.1287341

In [51]:
# Step 2: Helper Functions
# Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# Function to preprocess user input
def preprocess_text(text):
    words = text.lower().split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review

In [ ]:
### Prediction  function

def predict_sentiment(review):
    preprocessed_input=preprocess_text(review)

    prediction=model.predict(preprocessed_input)

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    
    return sentiment, prediction[0][0]

In [53]:
# Step 4: User Input and Prediction
# Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."

sentiment,score=predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Positive
Prediction Score: 0.642218828201294
